In [ ]:
!pip install pywavelets scipy opencv-python tqdm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pywt
import cv2

from scipy import signal
from tqdm import tqdm

import os

In [ ]:
import os

print(os.path.getsize("mitbih_train.csv")/(1024*1024))

In [ ]:
print(train_df.shape)

print(train_df.iloc[:5,-1])

print(train_df[187].value_counts())

In [ ]:
import os

for file in os.listdir():
    print(file)

In [ ]:
import pandas as pd

train_df = pd.read_csv("mitbih_train.csv")

print(train_df.shape)

train_df.head()

In [ ]:
train_df = pd.read_csv(
    "mitbih_train.csv",
    header=None
)

print(train_df.shape)

Class distribution check

In [ ]:
train_df[187].value_counts()

In [ ]:
X = train_df.iloc[:, :-1].values
y = train_df.iloc[:, -1].values

print(X.shape)
print(y.shape)

Vizualising ECG

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))

plt.plot(X[0])

plt.title("ECG Beat")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")

plt.show()

Visualize One Sample From Each Class

In [ ]:
classes = [0,1,2,3,4]

plt.figure(figsize=(15,10))

for i,c in enumerate(classes):

    idx = np.where(y == c)[0][0]

    plt.subplot(3,2,i+1)

    plt.plot(X[idx])

    plt.title(f"Class {c}")

plt.tight_layout()

plt.show()

Bandpass Filtering

In [ ]:
!pip install scipy

In [ ]:
from scipy import signal

In [ ]:
def bandpass_filter(ecg,
                    lowcut=0.5,
                    highcut=40,
                    fs=125,
                    order=4):

    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    b,a = signal.butter(
        order,
        [low,high],
        btype='band'
    )

    filtered = signal.filtfilt(
        b,
        a,
        ecg
    )

    return filtered

Testing filtering

In [ ]:
sample = X[0]

filtered = bandpass_filter(sample)

plt.figure(figsize=(14,5))

plt.plot(sample,label="Original")

plt.plot(filtered,label="Filtered")

plt.legend()

plt.show()

Wavelet library instalation

In [ ]:
!pip install PyWavelets

Generating CWT Scalogram

In [ ]:
import pywt

In [ ]:
def create_scalogram(signal_data):

    scales = np.arange(1,128)

    coefficients, frequencies = pywt.cwt(
        signal_data,
        scales,
        'morl'
    )

    return coefficients

In [ ]:
coef = create_scalogram(filtered)

plt.figure(figsize=(8,6))

plt.imshow(
    coef,
    aspect='auto',
    cmap='jet'
)

plt.title("CWT Scalogram")

plt.colorbar()

plt.show()

*ResNet will learn from this

Converting scalogram to rgb

In [ ]:
!pip install opencv-python

In [ ]:
import cv2
import numpy as np

def scalogram_to_rgb(coef):

    coef = np.abs(coef)

    coef = (coef - coef.min()) / (coef.max() - coef.min())

    coef = (coef * 255).astype(np.uint8)

    rgb = cv2.applyColorMap(
        coef,
        cv2.COLORMAP_JET
    )

    rgb = cv2.resize(
        rgb,
        (224,224)
    )

    return rgb

testing

In [ ]:
rgb = scalogram_to_rgb(coef)

import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))

plt.imshow(
    cv2.cvtColor(
        rgb,
        cv2.COLOR_BGR2RGB
    )
)

plt.axis("off")

plt.show()

Creating Class Folders

In [ ]:
import os

classes = {
    0:"normal",
    1:"supraventricular",
    2:"ventricular",
    3:"fusion",
    4:"unknown"
}

for c in classes.values():
    os.makedirs(
        f"scalograms/{c}",
        exist_ok=True
    )

Verifying Labels

In [ ]:
train_df[187].value_counts()

Creating a balanced subdata set

In [ ]:
import pandas as pd

balanced_df = pd.concat([
    train_df[train_df[187] == 0].sample(641, random_state=42),
    train_df[train_df[187] == 1].sample(641, random_state=42),
    train_df[train_df[187] == 2].sample(641, random_state=42),
    train_df[train_df[187] == 3],
    train_df[train_df[187] == 4].sample(641, random_state=42)
])

balanced_df = balanced_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(balanced_df.shape)

Extracting features and labels

In [ ]:
X_bal = balanced_df.iloc[:,:-1].values
y_bal = balanced_df.iloc[:,-1].values

Creating scalogram dset

In [ ]:
import os

classes = {
    0:"normal",
    1:"supraventricular",
    2:"ventricular",
    3:"fusion",
    4:"unknown"
}

for folder in classes.values():

    os.makedirs(
        f"scalograms/{folder}",
        exist_ok=True
    )

Generating Images

In [ ]:
from tqdm import tqdm

for idx in tqdm(range(len(X_bal))):

    signal_data = X_bal[idx]

    label = int(y_bal[idx])

    filtered = bandpass_filter(
        signal_data
    )

    coef = create_scalogram(
        filtered
    )

    rgb = scalogram_to_rgb(
        coef
    )

    class_name = classes[label]

    cv2.imwrite(
        f"scalograms/{class_name}/{idx}.png",
        rgb
    )

In [ ]:
import os

for c in classes.values():

    print(
        c,
        len(os.listdir(
            f"scalograms/{c}"
        ))
    )

Phase 2

In [ ]:
!pip install torch torchvision scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets
from torchvision import transforms
from torchvision import models

from torch.utils.data import DataLoader
from torch.utils.data import random_split

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
import os

os.listdir("scalograms")

In [ ]:
dataset = datasets.ImageFolder(
    root="scalograms"
)

print(dataset.classes)
print(len(dataset))

In [ ]:
P2:  Production Training Pipeline

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets
from torchvision import transforms
from torchvision import models

from torch.utils.data import DataLoader
from torch.utils.data import random_split

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda")

print(device)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
dataset = datasets.ImageFolder(
    root="scalograms",
    transform=transform
)

print(dataset.classes)

print(len(dataset))

In [ ]:
train_size = int(
    0.8 * len(dataset)
)

val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size,val_size]
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

Loading Resnet50

In [ ]:
model = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

Replacing Final Layer

In [ ]:
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    5
)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

Training with accuracy tracking

In [ ]:
EPOCHS = 10

train_losses = []

val_accuracies = []

best_acc = 0

In [ ]:
for epoch in range(EPOCHS):

    model.train()

    running_loss = 0

    correct = 0

    total = 0

    for images,labels in train_loader:

        images = images.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _,preds = torch.max(
            outputs,
            1
        )

        total += labels.size(0)

        correct += (
            preds == labels
        ).sum().item()

    train_acc = 100 * correct / total

    train_losses.append(
        running_loss
    )

    model.eval()

    correct = 0

    total = 0

    with torch.no_grad():

        for images,labels in val_loader:

            images = images.to(device)

            labels = labels.to(device)

            outputs = model(images)

            _,preds = torch.max(
                outputs,
                1
            )

            total += labels.size(0)

            correct += (
                preds == labels
            ).sum().item()

    val_acc = 100 * correct / total

    val_accuracies.append(
        val_acc
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
        f" | Loss: {running_loss:.4f}"
        f" | Train Acc: {train_acc:.2f}%"
        f" | Val Acc: {val_acc:.2f}%"
    )

    if val_acc > best_acc:

        best_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_resnet50.pth"
        )

Plot Training

In [ ]:
plt.figure(figsize=(12,4))

plt.plot(train_losses)

plt.title("Training Loss")

plt.show()

In [ ]:
model.load_state_dict(
    torch.load(
        "best_resnet50.pth"
    )
)

In [ ]:
model.eval()

predictions = []

actuals = []

with torch.no_grad():

    for images,labels in val_loader:

        images = images.to(device)

        outputs = model(images)

        preds = torch.argmax(
            outputs,
            dim=1
        )

        predictions.extend(
            preds.cpu().numpy()
        )

        actuals.extend(
            labels.numpy()
        )

In [ ]:
print(
    classification_report(
        actuals,
        predictions,
        target_names=dataset.classes
    )
)

In [ ]:
cm = confusion_matrix(
    actuals,
    predictions
)

print(cm)

Phase 2 done ! acheived val.acc of 90.1%. Saving and downloading model

In [ ]:
torch.save(
    model.state_dict(),
    "ecg_arrhythmia_resnet50.pth"
)

Phase 3 XAI

In [ ]:
!pip install grad-cam

In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

from torchvision import transforms
from torchvision import models

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
model = models.resnet50()

num_features = model.fc.in_features

model.fc = torch.nn.Linear(
    num_features,
    5
)

model.load_state_dict(
    torch.load(
        "best_resnet50.pth"
    )
)

model = model.to(device)

model.eval()

Picking Test img

In [ ]:
import os

sample_path = (
    "scalograms/ventricular/"
    + os.listdir(
        "scalograms/ventricular"
    )[0]
)

print(sample_path)

In [ ]:
img = Image.open(
    sample_path
).convert("RGB")

img_np = np.array(img)

img_np = img_np.astype(
    np.float32
)/255

Transforming

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

input_tensor = transform(
    img
).unsqueeze(0).to(device)

Prediction

In [ ]:
classes = [
    "fusion",
    "normal",
    "supraventricular",
    "unknown",
    "ventricular"
]

with torch.no_grad():

    output = model(
        input_tensor
    )

pred = torch.argmax(
    output,
    dim=1
)

print(
    classes[
        pred.item()
    ]
)

Target Layer for Resnet50

In [ ]:
target_layers = [
    model.layer4[-1]
]

Generating  grad-cam and  overlay heatmap

In [ ]:
cam = GradCAM(
    model=model,
    target_layers=target_layers
)

grayscale_cam = cam(
    input_tensor=input_tensor
)[0]

In [ ]:
visualization = show_cam_on_image(
    img_np,
    grayscale_cam,
    use_rgb=True
)

Display

In [ ]:
plt.figure(figsize=(12,6))

plt.subplot(1,2,1)

plt.imshow(img_np)

plt.title(
    "Original Scalogram"
)

plt.axis("off")

plt.subplot(1,2,2)

plt.imshow(
    visualization
)

plt.title(
    "Grad-CAM"
)

plt.axis("off")

plt.show()